# NTE lookup-key normalization benchmark

This benchmark tests normalization for the initial entity-resolution key.

MCN provides an attested name plus the expected GeoNames ID. Each MCN name is treated as lookup input.
A policy succeeds when the normalized key resolves back to the expected entity.

Policies are tested incrementally:

1. `exact`
2. `lower`
3. `casefold` — NFC + whitespace cleanup + Unicode casefold
4. `diacritic_fold` — casefold + NFD + removal of combining marks

Normalization is computed at runtime.


In [ ]:
import sqlite3
import collections
import unicodedata
import pandas as pd
import xml.etree.ElementTree as ET

from pathlib import Path
from dataclasses import dataclass
from enum import Enum

pd.set_option("display.max_rows", 1000)
pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 100)

PROJECT_ROOT  = Path.cwd().parent
MCN_PATH      = PROJECT_ROOT / "data" / "repos" / "more-cultural-names"
MCN_LOCATIONS = MCN_PATH / "locations.xml"
MCN_LANGUAGES = MCN_PATH / "languages.xml"
DB_PATH       = PROJECT_ROOT / "data" / "names.sqlite"


In [ ]:
def _clean_ws(text: str) -> str:
    return " ".join(text.strip().split())

def norm_lower(text: str) -> str:
    return _clean_ws(unicodedata.normalize("NFC", text)).lower()

def norm_casefold(text: str) -> str:
    return _clean_ws(unicodedata.normalize("NFC", text)).casefold()

def norm_diacritic_fold(text: str) -> str:
    folded = norm_casefold(text)
    decomposed = unicodedata.normalize("NFD", folded)
    stripped = "".join(ch for ch in decomposed if not unicodedata.category(ch).startswith("M"))
    return unicodedata.normalize("NFC", stripped)

def get_conn() -> sqlite3.Connection:
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    conn.execute("PRAGMA foreign_keys = ON;")
    conn.create_function("nte_lower", 1, norm_lower, deterministic=True)
    conn.create_function("nte_casefold", 1, norm_casefold, deterministic=True)
    conn.create_function("nte_diacritic_fold", 1, norm_diacritic_fold, deterministic=True)
    return conn

def run_query(sql: str, params: tuple = ()) -> pd.DataFrame:
    with get_conn() as conn:
        return pd.read_sql_query(sql, conn, params=params)

samples = ["Istanbul", "İstanbul", "Málaga", "MALAGA", "  New   York  ", "Straße"]
display(pd.DataFrame({
    "raw": samples,
    "lower": [norm_lower(x) for x in samples],
    "casefold": [norm_casefold(x) for x in samples],
    "diacritic_fold": [norm_diacritic_fold(x) for x in samples],
}))


## Load MCN benchmark data

This mirrors the general benchmark. The GeoNames ID attached to each MCN name is treated as the expected
resolved entity.


In [ ]:
languages_tree = ET.parse(MCN_LANGUAGES)

@dataclass(frozen=True)
class LangInfo:
    mcn_id: str
    iso1: str | None
    iso2: str | None
    iso3: str | None

    @property
    def iso_codes(self) -> tuple[str, ...]:
        return tuple(dict.fromkeys(c for c in (self.iso1, self.iso2, self.iso3) if c))

lang_info = {}
for lang in languages_tree.getroot().findall("Language"):
    mcn_id = (lang.findtext("Id") or "").strip()
    if not mcn_id:
        continue
    node = lang.find("Code")
    iso1 = iso2 = iso3 = None
    if node is not None:
        iso1 = node.attrib.get("iso-639-1") or None
        iso2 = node.attrib.get("iso-639-2") or None
        iso3 = node.attrib.get("iso-639-3") or None
    lang_info[mcn_id] = LangInfo(mcn_id, iso1, iso2, iso3)

locations_tree = ET.parse(MCN_LOCATIONS)
ck3_names = {}

for loc in locations_tree.iter("LocationEntity"):
    geonames_tag = loc.find("GeoNamesId")
    if geonames_tag is None:
        continue

    geonames_id = int(geonames_tag.text)
    names = {n.attrib["language"]: n.attrib["value"] for n in loc.findall("Names/Name")}
    if not names:
        continue

    for game_id in loc.findall("GameIds/GameId"):
        if game_id.attrib.get("game") != "CK3":
            continue
        title = game_id.text.strip() if game_id.text is not None else ""
        if title.startswith("d_nf") or title.startswith("b_"):
            continue
        ck3_names[(title, geonames_id)] = names

attested_rows = [
    (title, gid, lang, name)
    for (title, gid), names in ck3_names.items()
    for lang, name in names.items()
]

ck3_geonames_ids = sorted({gid for _, gid in ck3_names})

print(f"CK3 title/entity pairs : {len(ck3_names)}")
print(f"distinct entities      : {len(ck3_geonames_ids)}")
print(f"attested input rows    : {len(attested_rows)}")


## Cache NTE names for the benchmark entities

For initial lookup resolution, the candidate surfaces include:

- `geoname.name`
- `alternate_name.alternate_name`
- linked `wikidata_location_name.name`

SQLite computes the experimental normalization keys at runtime.


In [ ]:
@dataclass(frozen=True)
class NameSurface:
    source: str
    surface_type: str
    geonames_id: int
    name: str
    lower_key: str
    casefold_key: str
    diacritic_key: str
    stored_norm: str | None

def get_nte_surfaces_for_geonames_id(gid: int) -> list[NameSurface]:
    df = run_query("""
        SELECT
            'geonames' AS source,
            'primary' AS surface_type,
            g.geonameid AS geonames_id,
            g.name AS name,
            nte_lower(g.name) AS lower_key,
            nte_casefold(g.name) AS casefold_key,
            nte_diacritic_fold(g.name) AS diacritic_key,
            NULL AS stored_norm
        FROM geoname AS g
        WHERE g.geonameid = ?

        UNION ALL

        SELECT
            'geonames' AS source,
            'alternate' AS surface_type,
            a.geonameid AS geonames_id,
            a.alternate_name AS name,
            nte_lower(a.alternate_name) AS lower_key,
            nte_casefold(a.alternate_name) AS casefold_key,
            nte_diacritic_fold(a.alternate_name) AS diacritic_key,
            a.normalized_name AS stored_norm
        FROM alternate_name AS a
        WHERE a.geonameid = ?

        UNION ALL

        SELECT
            'wikidata' AS source,
            n.term_type AS surface_type,
            CAST(g.geonames_id AS INTEGER) AS geonames_id,
            n.name AS name,
            nte_lower(n.name) AS lower_key,
            nte_casefold(n.name) AS casefold_key,
            nte_diacritic_fold(n.name) AS diacritic_key,
            n.name_norm AS stored_norm
        FROM wikidata_location_geonames AS g
        JOIN wikidata_location_name AS n ON n.qid = g.qid
        WHERE g.geonames_id = CAST(? AS TEXT)
    """, (gid, gid, gid))

    return [
        NameSurface(
            row.source,
            row.surface_type,
            int(row.geonames_id),
            row.name,
            row.lower_key,
            row.casefold_key,
            row.diacritic_key,
            row.stored_norm,
        )
        for row in df.itertuples(index=False)
    ]

nte_cache = {gid: get_nte_surfaces_for_geonames_id(gid) for gid in ck3_geonames_ids}

print(f"entities cached     : {len(nte_cache)}")
print(f"total name surfaces : {sum(len(v) for v in nte_cache.values())}")


## Build resolution indexes

Each normalization key maps to all benchmark GeoNames entities carrying that key. This lets the benchmark
measure both recall and ambiguity.


In [ ]:
def build_index(key_fn):
    entity_index = collections.defaultdict(set)
    surface_index = collections.defaultdict(list)

    for gid, surfaces in nte_cache.items():
        for s in surfaces:
            key = key_fn(s)
            entity_index[key].add(gid)
            surface_index[key].append(s)

    return dict(entity_index), dict(surface_index)

exact_index, exact_surfaces = build_index(lambda s: s.name)
lower_index, lower_surfaces = build_index(lambda s: s.lower_key)
casefold_index, casefold_surfaces = build_index(lambda s: s.casefold_key)
diacritic_index, diacritic_surfaces = build_index(lambda s: s.diacritic_key)

print("distinct keys:")
print(f"  exact          {len(exact_index)}")
print(f"  lower          {len(lower_index)}")
print(f"  casefold       {len(casefold_index)}")
print(f"  diacritic_fold {len(diacritic_index)}")


In [ ]:
class MatchStatus(Enum):
    EXACT = "exact"
    LOWER = "lower"
    CASEFOLD = "casefold"
    DIACRITIC_FOLD = "diacritic_fold"
    WRONG_ONLY = "wrong_only"
    NO_MATCH = "no_match"

@dataclass
class Result:
    title: str
    geonames_id: int
    mcn_lang: str
    input_name: str
    status: MatchStatus
    matched_entity_ids: tuple[int, ...]
    matched_names: tuple[str, ...]
    matched_sources: tuple[str, ...]
    ambiguous: bool

def compare_one(title, gid, mcn_lang, input_name):
    ladder = [
        (MatchStatus.EXACT, input_name, exact_index, exact_surfaces),
        (MatchStatus.LOWER, norm_lower(input_name), lower_index, lower_surfaces),
        (MatchStatus.CASEFOLD, norm_casefold(input_name), casefold_index, casefold_surfaces),
        (MatchStatus.DIACRITIC_FOLD, norm_diacritic_fold(input_name), diacritic_index, diacritic_surfaces),
    ]

    for status, key, idx, surfaces in ladder:
        ids = tuple(sorted(idx.get(key, set())))
        if gid in ids:
            rows = [s for s in surfaces.get(key, []) if s.geonames_id == gid]
            return Result(
                title, gid, mcn_lang, input_name, status, ids,
                tuple(dict.fromkeys(s.name for s in rows)),
                tuple(sorted({s.source for s in rows})),
                len(ids) > 1,
            )

    broad_key = norm_diacritic_fold(input_name)
    ids = tuple(sorted(diacritic_index.get(broad_key, set())))
    status = MatchStatus.WRONG_ONLY if ids else MatchStatus.NO_MATCH
    return Result(title, gid, mcn_lang, input_name, status, ids, (), (), False)

results = [compare_one(*row) for row in attested_rows]
print(f"comparisons run: {len(results)}")


# Global results

The status counts are incremental, so `diacritic_fold` means that all less aggressive policies failed first.


In [ ]:
counts = collections.Counter(r.status for r in results)
n = len(results)

for status in MatchStatus:
    c = counts[status]
    print(f"{status.value:16s} {c:7d} ({c/n:.2%})")

resolved_tiers = [
    MatchStatus.EXACT,
    MatchStatus.LOWER,
    MatchStatus.CASEFOLD,
    MatchStatus.DIACRITIC_FOLD,
]

cum = 0
print()
for status in resolved_tiers:
    cum += counts[status]
    print(f"resolved through {status.value:16s}: {cum:7d}/{n} = {cum/n:.2%}")

before_diacritic = counts[MatchStatus.DIACRITIC_FOLD] + counts[MatchStatus.WRONG_ONLY] + counts[MatchStatus.NO_MATCH]
if before_diacritic:
    rescued = counts[MatchStatus.DIACRITIC_FOLD]
    print()
    print(f"diacritic-fold rescue among prior misses: {rescued}/{before_diacritic} = {rescued/before_diacritic:.2%}")


## Ambiguity cost


In [ ]:
rows = []
for status in resolved_tiers:
    subset = [r for r in results if r.status == status]
    ambiguous = sum(r.ambiguous for r in subset)
    rows.append({
        "tier": status.value,
        "resolved_rows": len(subset),
        "ambiguous_rows": ambiguous,
        "ambiguity_rate": ambiguous / len(subset) if subset else 0.0,
        "mean_entities": (
            sum(len(r.matched_entity_ids) for r in subset) / len(subset)
            if subset else 0.0
        ),
    })

ambiguity_df = pd.DataFrame(rows)
display(ambiguity_df.style.format({
    "ambiguity_rate": "{:.2%}",
    "mean_entities": "{:.3f}",
}))


## Examples rescued by each normalization tier


In [ ]:
def examples(status, limit=100):
    return pd.DataFrame([
        {
            "title": r.title,
            "geonames_id": r.geonames_id,
            "language": r.mcn_lang,
            "input": r.input_name,
            "matched_db_names": list(r.matched_names),
            "sources": list(r.matched_sources),
            "entity_count": len(r.matched_entity_ids),
        }
        for r in results if r.status == status
    ]).head(limit)

print("lower:")
display(examples(MatchStatus.LOWER))

print("casefold:")
display(examples(MatchStatus.CASEFOLD))

print("diacritic_fold:")
display(examples(MatchStatus.DIACRITIC_FOLD))


## Per-language gains


In [ ]:
by_lang = collections.defaultdict(list)
for r in results:
    by_lang[r.mcn_lang].append(r)

rows = []
for lang, rs in by_lang.items():
    c = collections.Counter(r.status for r in rs)
    attested = len(rs)
    resolved = sum(c[s] for s in resolved_tiers)
    rows.append({
        "mcn_lang": lang,
        "attested": attested,
        "exact": c[MatchStatus.EXACT],
        "lower_gain": c[MatchStatus.LOWER],
        "casefold_gain": c[MatchStatus.CASEFOLD],
        "diacritic_gain": c[MatchStatus.DIACRITIC_FOLD],
        "resolved_rate": resolved / attested if attested else 0.0,
    })

per_lang_df = pd.DataFrame(rows).sort_values(
    ["diacritic_gain", "attested"], ascending=[False, False]
).reset_index(drop=True)

display(per_lang_df.head(100).style.format({"resolved_rate": "{:.1%}"}))


## Collision rate inside the CK3 benchmark universe

This is not a full-database collision scan. It is intentionally limited to the entities already present in
the benchmark, so it remains cheap while still showing whether a normalization policy collapses many distinct
place names together.


In [ ]:
def collision_summary(index, label):
    ambiguous = [ids for ids in index.values() if len(ids) > 1]
    return {
        "policy": label,
        "distinct_keys": len(index),
        "ambiguous_keys": len(ambiguous),
        "ambiguous_key_rate": len(ambiguous) / len(index) if index else 0.0,
        "max_entities_per_key": max((len(ids) for ids in index.values()), default=0),
    }

collision_df = pd.DataFrame([
    collision_summary(exact_index, "exact"),
    collision_summary(lower_index, "lower"),
    collision_summary(casefold_index, "casefold"),
    collision_summary(diacritic_index, "diacritic_fold"),
])

display(collision_df.style.format({"ambiguous_key_rate": "{:.2%}"}))

collision_examples = []
for key, ids in diacritic_index.items():
    if len(ids) <= 1:
        continue
    collision_examples.append({
        "key": key,
        "entity_count": len(ids),
        "entity_ids": sorted(ids),
        "raw_names": sorted({s.name for s in diacritic_surfaces[key]})[:30],
    })

display(
    pd.DataFrame(collision_examples)
    .sort_values(["entity_count", "key"], ascending=[False, True])
    .head(100)
)


## Compare against the normalized fields already stored

GeoNames `alternate_name.normalized_name` and Wikidata `wikidata_location_name.name_norm` are compared
against the experimental runtime keys for the benchmarked rows.


In [ ]:
stored_rows = [
    s
    for surfaces in nte_cache.values()
    for s in surfaces
    if s.stored_norm is not None
]

audit = []
for source in ("geonames", "wikidata"):
    rows = [s for s in stored_rows if s.source == source and s.stored_norm != ""]
    n_rows = len(rows)
    if not n_rows:
        continue

    audit.append({
        "source": source,
        "rows": n_rows,
        "equals_lower": sum(s.stored_norm == s.lower_key for s in rows) / n_rows,
        "equals_casefold": sum(s.stored_norm == s.casefold_key for s in rows) / n_rows,
        "equals_diacritic_fold": sum(s.stored_norm == s.diacritic_key for s in rows) / n_rows,
    })

display(pd.DataFrame(audit).style.format({
    "equals_lower": "{:.1%}",
    "equals_casefold": "{:.1%}",
    "equals_diacritic_fold": "{:.1%}",
}))


## Wrong-only broad matches


In [ ]:
wrong_only_df = pd.DataFrame([
    {
        "title": r.title,
        "expected_geonames_id": r.geonames_id,
        "language": r.mcn_lang,
        "input": r.input_name,
        "diacritic_key": norm_diacritic_fold(r.input_name),
        "resolved_entity_ids": list(r.matched_entity_ids),
    }
    for r in results if r.status == MatchStatus.WRONG_ONLY
])

print(f"wrong-only rows: {len(wrong_only_df)}")
display(wrong_only_df.head(200))


In [ ]:
# Full-database collision audit:
# conservative key (casefold) vs diacritic-insensitive folded key

with get_conn() as conn:
    conn.execute("DROP TABLE IF EXISTS temp.lookup_collision_keys")

    # Materialize once so the expensive runtime normalization is not repeated
    # for every diagnostic query below.
    conn.execute("""
        CREATE TEMP TABLE lookup_collision_keys AS

        SELECT
            'geonames' AS source,
            CAST(geonameid AS TEXT) AS entity_id,
            name AS raw_name,
            nte_casefold(name) AS norm_key,
            nte_diacritic_fold(name) AS folded_key
        FROM geoname
        WHERE name IS NOT NULL AND name <> ''

        UNION ALL

        SELECT
            'geonames' AS source,
            CAST(geonameid AS TEXT) AS entity_id,
            alternate_name AS raw_name,
            nte_casefold(alternate_name) AS norm_key,
            nte_diacritic_fold(alternate_name) AS folded_key
        FROM alternate_name
        WHERE alternate_name IS NOT NULL AND alternate_name <> ''

        UNION ALL

        SELECT
            'wikidata' AS source,
            qid AS entity_id,
            name AS raw_name,
            nte_casefold(name) AS norm_key,
            nte_diacritic_fold(name) AS folded_key
        FROM wikidata_location_name
        WHERE name IS NOT NULL AND name <> ''
    """)

    conn.execute("""
        CREATE INDEX temp.idx_collision_norm
        ON lookup_collision_keys(source, norm_key, entity_id)
    """)

    conn.execute("""
        CREATE INDEX temp.idx_collision_folded
        ON lookup_collision_keys(source, folded_key, entity_id)
    """)

    # Overall collision rates for conservative and diacritic-insensitive keys.
    summary_df = pd.read_sql_query("""
        WITH
        norm_counts AS (
            SELECT
                source,
                norm_key AS key,
                COUNT(DISTINCT entity_id) AS entity_count
            FROM lookup_collision_keys
            WHERE norm_key <> ''
            GROUP BY source, norm_key
        ),
        folded_counts AS (
            SELECT
                source,
                folded_key AS key,
                COUNT(DISTINCT entity_id) AS entity_count
            FROM lookup_collision_keys
            WHERE folded_key <> ''
            GROUP BY source, folded_key
        ),
        summary AS (
            SELECT
                source,
                'casefold' AS policy,
                COUNT(*) AS distinct_keys,
                SUM(entity_count > 1) AS ambiguous_keys,
                MAX(entity_count) AS max_entities_per_key,
                AVG(entity_count) AS mean_entities_per_key
            FROM norm_counts
            GROUP BY source

            UNION ALL

            SELECT
                source,
                'diacritic_fold' AS policy,
                COUNT(*) AS distinct_keys,
                SUM(entity_count > 1) AS ambiguous_keys,
                MAX(entity_count) AS max_entities_per_key,
                AVG(entity_count) AS mean_entities_per_key
            FROM folded_counts
            GROUP BY source
        )
        SELECT
            *,
            CAST(ambiguous_keys AS REAL) / distinct_keys AS ambiguous_key_rate
        FROM summary
        ORDER BY source, policy
    """, conn)

    display(summary_df.style.format({
        "ambiguous_key_rate": "{:.3%}",
        "mean_entities_per_key": "{:.4f}",
    }))

    # Keys that become ambiguous specifically because diacritics were removed.
    # A "new collision" means:
    #   - the folded key maps to >1 entity
    #   - every conservative key contributing to it was individually unambiguous
    newly_ambiguous_df = pd.read_sql_query("""
        WITH norm_counts AS (
            SELECT
                source,
                norm_key,
                COUNT(DISTINCT entity_id) AS entity_count
            FROM lookup_collision_keys
            WHERE norm_key <> ''
            GROUP BY source, norm_key
        ),
        folded_groups AS (
            SELECT
                source,
                folded_key,
                COUNT(DISTINCT entity_id) AS folded_entity_count,
                COUNT(DISTINCT norm_key) AS conservative_key_count,
                MAX(
                    (
                        SELECT nc.entity_count
                        FROM norm_counts nc
                        WHERE nc.source = k.source
                          AND nc.norm_key = k.norm_key
                    )
                ) AS max_conservative_entity_count
            FROM lookup_collision_keys k
            WHERE folded_key <> ''
            GROUP BY source, folded_key
        )
        SELECT
            source,
            COUNT(*) AS newly_ambiguous_keys,
            MAX(folded_entity_count) AS max_entities_in_new_collision,
            AVG(folded_entity_count) AS mean_entities_in_new_collision
        FROM folded_groups
        WHERE folded_entity_count > 1
          AND max_conservative_entity_count = 1
        GROUP BY source
        ORDER BY source
    """, conn)

    print("Collisions introduced specifically by diacritic stripping:")
    display(newly_ambiguous_df)

    # Largest newly-created collisions, with example spellings and entity IDs.
    collision_examples_df = pd.read_sql_query("""
        WITH norm_counts AS (
            SELECT
                source,
                norm_key,
                COUNT(DISTINCT entity_id) AS entity_count
            FROM lookup_collision_keys
            WHERE norm_key <> ''
            GROUP BY source, norm_key
        ),
        folded_groups AS (
            SELECT
                k.source,
                k.folded_key,
                COUNT(DISTINCT k.entity_id) AS folded_entity_count,
                COUNT(DISTINCT k.norm_key) AS conservative_key_count,
                MAX(nc.entity_count) AS max_conservative_entity_count
            FROM lookup_collision_keys k
            JOIN norm_counts nc
              ON nc.source = k.source
             AND nc.norm_key = k.norm_key
            WHERE k.folded_key <> ''
            GROUP BY k.source, k.folded_key
        )
        SELECT
            fg.source,
            fg.folded_key,
            fg.folded_entity_count AS entity_count,
            fg.conservative_key_count,
            GROUP_CONCAT(DISTINCT k.raw_name) AS raw_names,
            GROUP_CONCAT(DISTINCT k.entity_id) AS entity_ids
        FROM folded_groups fg
        JOIN lookup_collision_keys k
          ON k.source = fg.source
         AND k.folded_key = fg.folded_key
        WHERE fg.folded_entity_count > 1
          AND fg.max_conservative_entity_count = 1
        GROUP BY
            fg.source,
            fg.folded_key,
            fg.folded_entity_count,
            fg.conservative_key_count
        ORDER BY fg.folded_entity_count DESC, fg.source, fg.folded_key
        LIMIT 100
    """, conn)

    print("Largest collisions introduced by diacritic stripping:")
    display(collision_examples_df)

    # How often stripping marks actually changes a stored name key.
    changed_df = pd.read_sql_query("""
        SELECT
            source,
            COUNT(*) AS name_rows,
            SUM(norm_key <> folded_key) AS changed_rows,
            CAST(SUM(norm_key <> folded_key) AS REAL) / COUNT(*) AS changed_rate
        FROM lookup_collision_keys
        GROUP BY source
        ORDER BY source
    """, conn)

    print("How much of the database is affected by diacritic stripping:")
    display(changed_df.style.format({"changed_rate": "{:.2%}"}))